# **1. Problem Statement**


## 1.1 Business Context


A sales forecast is a prediction of future sales revenue based on historical data, industry trends, and the status of the current sales pipeline. Businesses use the sales forecast to estimate weekly, monthly, quarterly, and annual sales totals. A company needs to make an accurate sales forecast as it adds value across an organization and helps the different verticals to chalk out their future course of action.

Forecasting helps an organization plan its sales operations by region and provides valuable insights to the supply chain team regarding the procurement of goods and materials. An accurate sales forecast process has many benefits which include improved decision-making about the future and reduction of sales pipeline and forecast risks. Moreover, it helps to reduce the time spent in planning territory coverage and establish benchmarks that can be used to assess trends in the future.

## 1.2 Objective


SuperKart is a retail chain operating supermarkets and food marts across various tier cities, offering a wide range of products. To optimize its inventory management and make informed decisions around regional sales strategies, SuperKart wants to accurately forecast the sales revenue of its outlets for the upcoming quarter.

To operationalize these insights at scale, the company has partnered with a data science firm to develop and deploy a robust forecasting solution that can be integrated into SuperKart’s decision-making systems and used across its network of stores.

## 1.3 Data Description


The data contains the different attributes of the various products and stores. The detailed data dictionary is given below.

- **Product_Id** - unique identifier of each product, each identifier having two letters at the beginning followed by a number.
- **Product_Weight** - weight of each product
- **Product_Sugar_Content** - sugar content of each product like low sugar, regular and no sugar
- **Product_Allocated_Area** - ratio of the allocated display area of each product to the total display area of all the products in a store
- **Product_Type** - broad category for each product
- **Product_MRP** - maximum retail price of each product
- **Store_Id** - unique identifier of each store
- **Store_Establishment_Year** - year in which the store was established
- **Store_Size** - size of the store depending on sq. feet like high, medium and low
- **Store_Location_City_Type** - type of city in which the store is located like Tier 1, Tier 2 and Tier 3.
- **Store_Type** - type of store depending on the products that are being sold there
- **Product_Store_Sales_Total** - total revenue generated by the sale of that particular product in that particular store


# **2. Installing and Importing the necessary libraries**


In [ ]:
%pip install pandas scikit-learn xgboost huggingface_hub joblib


In [ ]:
import os
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from huggingface_hub import HfApi, login
import joblib
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, r2_score


# **3. Data Loading and Hugging Face Registration**


In [ ]:
# Authenticate with Hugging Face using the provided token
print('Logging into Hugging Face...')
import os
token = os.getenv('HF_TOKEN')
if token:
    login(token)
else:
    from huggingface_hub import notebook_login
    notebook_login()


In [ ]:
# Register the raw dataset to the Hugging Face dataset space first
api = HfApi()
print('Uploading raw dataset to Hugging Face dataset space...')
api.create_repo(repo_id='pvinayv/superkart-dataset', repo_type='dataset', exist_ok=True)
api.upload_file(
    path_or_fileobj='data/SuperKart.csv',
    path_in_repo='data/SuperKart.csv',
    repo_id='pvinayv/superkart-dataset',
    repo_type='dataset'
)
print('Raw dataset registered on Hugging Face dataset space.')


In [ ]:
# Load the dataset directly from the Hugging Face data space
raw_hf_url = 'https://huggingface.co/datasets/pvinayv/superkart-dataset/resolve/main/data/SuperKart.csv'
print(f'Loading dataset directly from Hugging Face: {raw_hf_url}')
df = pd.read_csv(raw_hf_url)
display(df.head())
print(f"Shape of the dataset: {df.shape}")
df.info()


# **4. Data Preparation & Split Registration**


In [ ]:
# 1. Handle Missing Values
df['Product_Weight'] = df['Product_Weight'].fillna(df['Product_Weight'].mean())
df['Store_Size'] = df['Store_Size'].fillna('Medium')

# 2. Standardize Categorical Variables (data cleaning)
df['Product_Sugar_Content'] = df['Product_Sugar_Content'].replace({'low fat': 'Low Fat', 'LF': 'Low Fat', 'reg': 'Regular'})

# 3. Train-Test Split
train, test = train_test_split(df, test_size=0.2, random_state=42)
train.to_csv('data/train.csv', index=False)
test.to_csv('data/test.csv', index=False)
print(f'Train shape: {train.shape}, Test shape: {test.shape}')


In [ ]:
# Upload resulting train and test datasets back to the Hugging Face data space
api.upload_file(path_or_fileobj='data/train.csv', path_in_repo='train.csv', repo_id='pvinayv/superkart-dataset', repo_type='dataset')
api.upload_file(path_or_fileobj='data/test.csv', path_in_repo='test.csv', repo_id='pvinayv/superkart-dataset', repo_type='dataset')
print('Processed splits uploaded back to Hugging Face successfully.')


# **5. Model Training and MLOps Deployment**


In [ ]:
# Load the train and test data directly from the Hugging Face data space
train_hf_url = 'https://huggingface.co/datasets/pvinayv/superkart-dataset/resolve/main/train.csv'
test_hf_url = 'https://huggingface.co/datasets/pvinayv/superkart-dataset/resolve/main/test.csv'

print(f'Loading training split from HF: {train_hf_url}')
train = pd.read_csv(train_hf_url)
print(f'Loading testing split from HF: {test_hf_url}')
test = pd.read_csv(test_hf_url)


In [ ]:
# Separate features (X) and target variable (y)
X_train = train.drop(columns=['Product_Store_Sales_Total', 'Product_Id', 'Store_Id'])
y_train = train['Product_Store_Sales_Total']
X_test = test.drop(columns=['Product_Store_Sales_Total', 'Product_Id', 'Store_Id'])
y_test = test['Product_Store_Sales_Total']

categorical_features = ['Product_Sugar_Content', 'Product_Type', 'Store_Size', 'Store_Location_City_Type', 'Store_Type']
numeric_features = ['Product_Weight', 'Product_Allocated_Area', 'Product_MRP', 'Store_Establishment_Year']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ])

# Define base model
base_model = XGBRegressor(random_state=42)

# Define parameters grid for tuning
param_grid = {
    'model__n_estimators': [50, 100],
    'model__max_depth': [3, 5],
    'model__learning_rate': [0.05, 0.1]
}

pipeline = Pipeline(steps=[ 
    ('preprocessor', preprocessor),
    ('model', base_model)
])

print('Tuning model with GridSearchCV...')
grid_search = GridSearchCV(pipeline, param_grid, cv=3, scoring='neg_root_mean_squared_error', n_jobs=-1)
grid_search.fit(X_train, y_train)
best_pipeline = grid_search.best_estimator_
print(f'Best parameters: {grid_search.best_params_}')


In [ ]:
# Evaluate model performance
y_pred = best_pipeline.predict(X_test)
rmse = mean_squared_error(y_test, y_pred) ** 0.5
r2 = r2_score(y_test, y_pred)
print(f'RMSE: {rmse:.2f}')
print(f'R2 Score: {r2:.4f}')


In [ ]:
# Serialize and Register the best model in Hugging Face model hub
joblib.dump(best_pipeline, 'xgboost_model.pkl')
api.create_repo(repo_id='pvinayv/superkart-model', repo_type='model', exist_ok=True)
api.upload_file(
    path_or_fileobj='xgboost_model.pkl',
    path_in_repo='xgboost_model.pkl',
    repo_id='pvinayv/superkart-model',
    repo_type='model'
)
print('Model registered in model hub successfully.')
